In [ ]:
from datetime import UTC, datetime

from astropy.table import Table
from lsst.rsp import get_tap_service, retrieve_query

In [ ]:
service = get_tap_service("tap")

In [ ]:
N_REQUESTS = 20

In [ ]:
query = """
        SELECT TOP 1000 * 
        FROM dp2.Object 
        """
results = service.run_async(query)

In [ ]:
# coords
coords_string = "objectId\tcoord_ra\tcoord_dec\n" + "\n".join(
    f"{row['objectId']}\t{row['coord_ra']}\t{row['coord_dec']}" 
    for _, row in enumerate(results)
)

In [ ]:
ut1 = Table.read(coords_string, format='ascii.basic')

In [ ]:
query = """
SELECT ut1.coord_ra AS ut1_ra, ut1.coord_dec AS ut1_dec,
    dp2.Object.objectId, dp2.Object.coord_ra, dp2.Object.coord_dec,
    dp2.Object.r_cModelMag
FROM dp2.Object
JOIN TAP_UPLOAD.ut1 AS ut1
ON DISTANCE(POINT('ICRS', dp2.Object.coord_ra, dp2.Object.coord_dec), POINT('ICRS', ut1.coord_ra, ut1.coord_dec)) < 0.00027
ORDER BY dp2.Object.objectId
"""

In [ ]:
for i in range(N_REQUESTS):
    job = service.submit_job(query, uploads={"ut1": ut1})
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    print('Job phase is', job.phase)
    print('Job ID is', job.job.jobid)
    results = job.fetch_result()
    results.to_table()
    assert job.phase == "COMPLETED"